# E7 — APTOS2019 optimization

Compare one trained checkpoint and its descendants on the same frozen validation sequence. [Optimization notebook index](README.md) · [HPC execution guide](../../novelty_integrated/optimization/README.md#staged-hpc-campaign) · [Historical backend and pruning audit](../../docs/OPTIMIZATION_COMPATIBILITY.md)

## Original checkpoint and provenance

| Receipt | Value |
| --- | --- |
| Original relative checkpoint | `foundation_e7/APTOS2019/run/train/best.pt` |
| Checkpoint SHA-256 | `5ef9816b9c7511a2b0347f7967543f04d70a77c9d0de630fd4116f714c38d11c` |
| Selected epoch | 19 |
| Frozen manifest SHA-256 | `73d1546e3c2fd2bf9e3d0fd10af914cb523d28e4bc8e1f25212af454d4be8854` |
| Validation images | 366 |
| Processor SHA-256 | `fd575b890da5a949493e1d1e7a70bfcb9e4b99fe444004d2dbfa253add254741` |
| Recorded architecture | `hf_v1_global_local` |

The published validation macro-F1 is **0.696492**. It documents checkpoint selection and is separate from new CPU FP32 measurements.
[Published validation evidence](../../results/APTOS2019/e7_global_local/validation_evidence.json)

## Study design

E7 retains its 384-pixel global view, two 224-pixel local crops, shared HF V1 backbone and original gate. The archived timm ConvNeXtV2 experiments use a different GRN layout and are methodological references. These notebooks exercise the receipt-matched HF V1 adapter.

| Stage | Treatment |
| --- | --- |
| Benchmark original | FP32 eager, TorchScript, ONNX Runtime CPU, OpenVINO CPU. Every export is gated on full-sequence numerical and prediction agreement. |
| Prune without recovery | Independent L1 expansion-channel removals of 10%, 30%, and 50%, rounded down per layer. Report the actual parameter and MAC reduction. |
| Recovery | **Recovery not supported** for this arm. Measure original and pruned variants under the same CPU allocation. |

Batch 1, full ordered validation by default, 20 warm-ups and three timed passes per backend and scope. CPU affinity and thread count are explicit. Model-only excludes image loading and transforms. End-to-end includes them. Quality uses five grades in order 0–4. FLOPs count two operations per counted Conv2d/Linear MAC and exclude normalization, elementwise operations and I/O. E7 counts shared parameters once and computes work for all three views. The validation split selected the parent checkpoints, so trade-offs are exploratory.

## Execution controls

Default execution dispatches no process. HPC stages require separate allocation arguments. HPC jobs verify the original bytes and write exclusively under the dedicated campaign root. No stage uses final test.

In [ ]:
JOB = 'E7_APTOS2019'
from pathlib import Path
import os
import subprocess
import sys

# Run All is read-only. To dispatch one stage, set it explicitly and supply approved arguments.
RUN_STAGE = None  # audit, prepare, measure, recover, measure-recovered
STAGE_OPTIONS = []  # e.g. allocation arguments from the HPC guide

REPO = Path(os.environ.get("CADVIT_REPOSITORY", Path.cwd())).expanduser().resolve()
REPO = next((p for p in (REPO, *REPO.parents)
             if (p / "novelty_integrated/optimization/hpc.py").is_file()), None)
if REPO is None:
    raise ValueError("Set CADVIT_REPOSITORY to the checkout containing the optimization framework.")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

if RUN_STAGE is not None:
    allowed = {"audit", "prepare", "measure", "recover", "measure-recovered"}
    if RUN_STAGE not in allowed:
        raise ValueError("Select a supported stage.")
    subprocess.run([sys.executable, "-m", "novelty_integrated.optimization.hpc", RUN_STAGE,
                    "--job", JOB, *STAGE_OPTIONS], cwd=REPO, check=True)


## Results and trade-offs

| Measurement | Status in this checkout |
| --- | --- |
| Original and pruning benchmark | `NOT_RUN` |
| Post-recovery benchmark | `NOT_SUPPORTED` |

Set `CADVIT_OPT_CAMPAIGN` to the external campaign root to display completed provenance-matched results. The display includes latency, throughput, model size, compute, aggregate and per-grade quality, confusion matrices, pruning/recovery deltas and publication figures.

In [ ]:
from IPython.display import display, Markdown, SVG
from novelty_integrated.optimization.notebook import load, render

CAMPAIGN = os.environ.get("CADVIT_OPT_CAMPAIGN")
QUALITY_BACKEND = "eager"  # Detailed per-grade metrics and confusion matrices for this backend.
reports = {}
for stage in ("measurement", "post-recovery"):
    report = load(JOB, CAMPAIGN, stage) if CAMPAIGN else None
    if report is None:
        display(Markdown(f"**{stage}** — NOT_RUN"))
        continue
    if reports and any(report[k] != previous[k] for previous in reports.values()
                       for k in ("protocol_sha256", "sample")):
        raise ValueError("Measurement and post-recovery reports are from different protocols or samples.")
    reports[stage] = report
    display(Markdown(f"### {stage}\n\n" + render(report, QUALITY_BACKEND)))
    for figure in ("comparison.svg", "tradeoff.svg"):
        path = Path(CAMPAIGN).expanduser() / JOB / stage / figure
        display(SVG(filename=str(path)))


## Current conclusion

No latency or pruning quality conclusion is available before a measured run. Compare the original and derived models within the same allocation and validation sequence. Report classifier agreement for backend changes and grade-level losses for pruning or recovery. Keep this exploratory validation comparison separate from final-test assessment.